In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from sklearn.metrics import classification_report, confusion_matrix

project_root = Path.cwd().resolve().parent.parent
processed_dir = project_root / 'src' / 'data' / 'processed'
models_dir = project_root / 'src' / 'models'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from src.models.model import PitStrategyNet

# Load data
X_test = np.load(processed_dir / 'X_test.npy')
y_test = np.load(processed_dir / 'y_test.npy')
meta = pd.read_csv(processed_dir / 'meta_test.csv')

with open(processed_dir / 'driver_mapping.json') as f:
    driver_mapping = {int(k): v for k, v in json.load(f).items()}

with open(processed_dir / 'compound_mapping.json') as f:
    compound_mapping = {int(k): v for k, v in json.load(f).items()}

print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

In [ ]:
# Load model
import yaml
from pathlib import Path

project_root = Path.cwd().resolve().parent.parent
with open(project_root / 'src' / 'config.yaml') as f:
    config = yaml.safe_load(f)

checkpoint = torch.load(models_dir / config['paths']['folder_name'] / config['paths']['model_filename'])
model = PitStrategyNet(
    input_dim=config['model']['input_dim'],
    hidden_dim_1=config['model']['hidden_dim_1'],
    hidden_dim_2=config['model']['hidden_dim_2'],
    dropout_rate=config['model']['dropout_rate'],
    num_classes=config['model']['num_classes']
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded: {config['paths']['model_filename']}")

In [ ]:
model.eval()
all_preds = []
all_labels = []
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

with torch.no_grad():
    for i in range(0, len(X_test_tensor), 32):
        batch = X_test_tensor[i:i+32]
        outputs = model(batch)
        preds = torch.argmax(outputs, dim=1)  # get predicted class 0, 1 or 2
        all_preds.append(preds.cpu())
        all_labels.append(torch.tensor(y_test[i:i+32]))

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print(classification_report(
    all_labels.numpy(),
    all_preds.numpy(),
    target_names=['HARD', 'MEDIUM', 'SOFT']
))
print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))

In [ ]:
# MC Dropout uncertainty over full test set
model.train()
results = []

with torch.no_grad():
    for idx in range(len(X_test_tensor)):
        x = X_test_tensor[idx].unsqueeze(0)
        preds = torch.stack([torch.softmax(model(x), dim=1) for _ in range(100)])
        mean = preds.mean(dim=0).squeeze().clone()
        std  = preds.std(dim=0).squeeze()
        predicted_class = mean.argmax().item()

        results.append({
            'driver': driver_mapping[int(meta.iloc[idx]['Driver'])],
            'lap': int(meta.iloc[idx]['LapNumber']),
            'pred_compound': compound_mapping[predicted_class],
            'actual_compound': compound_mapping[int(y_test[idx])],
            'confidence': mean[predicted_class].item(),
            'uncertainty': std[predicted_class].item()
        })

results_df = pd.DataFrame(results)
results_df['correct'] = (results_df['pred_compound'] == results_df['actual_compound']).astype(int)

results_df['uncertainty_band'] = pd.cut(results_df['uncertainty'],
    bins=[0, 0.1, 0.3, 1.0],
    labels=['Low', 'Medium', 'High']
)

print(results_df.sort_values('uncertainty', ascending=False).head(20))
print(results_df.groupby('uncertainty_band')['correct'].mean())
print(results_df.groupby('uncertainty_band')['correct'].count())

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

constrained_preds  = results_df['pred_compound'].map({'HARD': 0, 'MEDIUM': 1, 'SOFT': 2})
constrained_labels = results_df['actual_compound'].map({'HARD': 0, 'MEDIUM': 1, 'SOFT': 2})

print(classification_report(
    constrained_labels,
    constrained_preds,
    target_names=['HARD', 'MEDIUM', 'SOFT']
))
print(confusion_matrix(constrained_labels, constrained_preds))

In [ ]:
report = classification_report(
    all_labels.numpy(), all_preds.numpy(),
    target_names=['HARD', 'MEDIUM', 'SOFT'],
    output_dict=True
)

summary = {
    'experiment': config['paths']['folder_name'],
    'architecture': config['paths']['architecture'],
    'hidden_dim_1': config['model']['hidden_dim_1'],
    'hidden_dim_2': config['model']['hidden_dim_2'],
    'dropout_rate': config['model']['dropout_rate'],
    'learning_rate': config['training']['learning_rate'],
    'accuracy': report['accuracy'],
    'hard_f1': report['HARD']['f1-score'],
    'medium_f1': report['MEDIUM']['f1-score'],
    'soft_f1': report['SOFT']['f1-score'],
    'uq_low_acc': float(results_df.groupby(
        'uncertainty_band', observed=True)['correct'].mean().get('Low', 0)),
    'uq_med_acc': float(results_df.groupby(
        'uncertainty_band', observed=True)['correct'].mean().get('Medium', 0)),
    'uq_high_acc': float(results_df.groupby(
        'uncertainty_band', observed=True)['correct'].mean().get('High', 0)),
}

summary_path = project_root / 'src' / 'evaluation' / 'experiment_summary.json'

# Load existing or create new
if summary_path.exists():
    with open(summary_path) as f:
        all_summaries = json.load(f)
else:
    all_summaries = []

all_summaries.append(summary)

with open(summary_path, 'w') as f:
    json.dump(all_summaries, f, indent=2)

print("Summary saved.")